In [ ]:
# allow to sync with local code
%load_ext autoreload
%autoreload 2

In [ ]:
EXPERIMENT_IDS = {
    "random": {
        "CIFAR_10 - 10%": None,
        "Pascal_VOC - 10%": None,
        "Imdb - 10%": None,
    },
    "gold": {
        "CIFAR_10 - 10%": "354041301038171263",
        "Pascal_VOC - 10%": "251710914009724832",
        "Imdb - 10%": None,
    }
}

METRICS = ("test_auroc", "test_acc", "test_iou", "test_micro_iou")

MLFLOW_TRACKING_URI = "/home/ubuntu/goldener-examples/mlruns/"


In [ ]:
from mlflow import MlflowClient

client = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI
)


def get_metric_for_experiment(experiment_id, metric_names):
    """
    Get run name and specific metric value for all runs in an experiment.
    """
    runs = client.search_runs(experiment_ids=[experiment_id])

    run_data = []
    for run in runs:
        run_name = run.info.run_name
        random_shuffle_state = run.data.params["random_shuffle_state"]
        run_info = {"run_name": run_name, "seed": random_shuffle_state}
        metrics_info = {}
        for metric_name in metric_names:
            metric_value = run.data.metrics.get(metric_name, None)
            if metric_value is not None:
                metrics_info[metric_name] = metric_value
        if metrics_info:
            run_data.append(run_info | metrics_info)

    return run_data

In [ ]:
# --- Collect all data for all datasets ---
all_data = {}
for shuffling_type, experiment_ids in EXPERIMENT_IDS.items():
    all_data[shuffling_type] = {}
    for dataset_name, experiment_id in experiment_ids.items():
        if experiment_id is not None:
            all_data[shuffling_type][dataset_name] = get_metric_for_experiment(
                experiment_id,
                METRICS,
            )

all_data

In [ ]:
import pandas as pd

# --- Convert to flat DataFrame ---
rows = []
for shuffling_type, datasets in all_data.items():
    for dataset_name, runs in datasets.items():
        for run in runs:
            for metric in METRICS:
                if metric in run:
                    rows.append(
                        {
                            "shuffle": shuffling_type,
                            "dataset": dataset_name,
                            "run_name": run["run_name"],
                            "seed": str(run["seed"]),
                            "metric": metric,
                            "value": run[metric],
                        }
                    )

df = pd.DataFrame(rows)
df.head()

In [ ]:
metrics_present = [m for m in METRICS if m in df["metric"].values]

summary = df.groupby(["shuffle", "dataset", "metric"])["value"].agg(mean="mean", std="std").round(4)

# Build one row per dataset with "min / max" string per metric
table_rows = []
for shuffle, experiment_ids in EXPERIMENT_IDS.items():
    for dataset in experiment_ids.keys():
        row = {"shuffle": shuffle, "dataset": dataset}
        for metric in metrics_present:
            metric_row = f"{metric} (mean +/- std)"
            try:
                mean = summary.loc[(shuffle, dataset, metric), "mean"]
                std = summary.loc[(shuffle, dataset, metric), "std"]
                row[metric_row] = f"{mean} +/- {std}"
            except KeyError:
                row[metric_row] = "-"
        table_rows.append(row)

table_df = pd.DataFrame(table_rows).set_index("dataset")
print(table_df.to_markdown())